# Notebook 01b · EDA toàn bộ dataset ASL Citizen (metadata + chỉ mục + mẫu video)

Mục tiêu: cả nhóm có **một bộ bảng và biểu đồ** mô tả toàn bộ 83k video để chọn training pipeline.

| Phần | Phạm vi | Tải gì | Thời gian |
|---|---|---|---|
| **A** | **Toàn bộ** metadata (mọi video, gloss, signer) + kích thước từng file từ chỉ mục ZIP | vài MB (CSV + mục lục ZIP) | ~1–2 phút |
| **B** | Mẫu video phân tầng theo signer: fps, độ phân giải, độ dài, MediaPipe thấy tay | ~1 GB (mặc định 1,500 video) | vài phút – chục phút |
| **C** | Tự tạo `full_eda_report.md` + gói `full_eda_bundle.zip` để tải về | – | vài giây |

Nếu đã chạy notebook 01 (có `manifests/meta_all.csv` và `zip_index.csv`), phần A **không cần mạng**.
**Kaggle:** bật Internet nếu chưa có cache. Kết quả nằm ở `asl_outputs/full_eda/` (Kaggle) hoặc `outputs/metrics/full_eda/` (local).

## 1. Setup

In [ ]:
import sys, importlib, subprocess, shutil, tempfile
# Cài thư viện thiếu TRƯỚC khi import numpy/pandas (tránh phải restart kernel)
for _mod in ("remotezip", "tqdm", "mediapipe"):
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _mod])
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

def _find_root():
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / "configs" / "config.yaml").exists():
            return d
    raise FileNotFoundError("Chạy notebook từ bên trong repo đã clone (thiếu configs/config.yaml).")

ROOT = _find_root(); sys.path.insert(0, str(ROOT))

from src.data import asl_citizen as A
from src.data import eda as E
print("repo:", ROOT)

## 2. Cấu hình

In [ ]:
CFG = dict(
    sample_videos=1500,      # số video mẫu phần B (chia đều theo signer)
    max_sample_gb=3.0,       # chặn cứng dung lượng tải cho phần B
    workers=8, seed=42,
    run_probe=True, probe_frames=8,   # MediaPipe: tỉ lệ thấy tay theo signer
    run_part_b=True,         # False = chỉ làm phần A (không tải video)
)
P = A.resolve_paths(ROOT)
OUT = P.metrics / "full_eda"; PLT = OUT / "plots"
OUT.mkdir(parents=True, exist_ok=True); PLT.mkdir(parents=True, exist_ok=True)
COL = {"train": "#4C78A8", "val": "#F58518", "test": "#54A24B"}
def save_tbl(df, name, index=True): df.to_csv(OUT / f"{name}.csv", index=index)
def save_fig(fig, name): fig.tight_layout(); fig.savefig(PLT / f"{name}.png", dpi=130)
print("kết quả →", OUT)

# Phần A — Toàn bộ metadata
## A0. Nạp bảng + kiểm tra chất lượng

In [ ]:
meta, idx, SRC = E.load_full_tables(P)
m = E.enrich(meta, idx)
m.to_csv(OUT / "meta_full.csv", index=False)          # bảng gốc đã gộp, cả nhóm dùng chung
print("nguồn:", SRC, "| shape:", m.shape, "| cột:", list(m.columns))
chk = {"số dòng": len(m), "video_id trùng": int(m.video_id.duplicated().sum()),
       "không khớp file trong zip": int(m.zip_name.isna().sum()), "thiếu file_size": int(m.file_size.isna().sum()),
       "signer nằm trong >1 split": int((m.groupby("signer_id").split.nunique() > 1).sum()),
       "gloss rỗng": int((m.gloss.str.len() == 0).sum())}
display(pd.Series(chk, name="giá trị").to_frame())
display(m.head(3))

## A1. Tổng quan theo split

In [ ]:
ov = E.split_overview(m); display(ov); save_tbl(ov, "A1_split_overview")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, c, t in zip(ax, ["videos", "signers", "gb"], ["Số video", "Số signer", "Dung lượng (GB)"]):
    ov.loc[list(E.SPLITS), c].plot.bar(ax=a, color=list(COL.values())); a.set_title(t); a.tick_params(axis="x", rotation=0)
save_fig(fig, "A1_split_overview"); plt.show()
print("Tỉ lệ video:", (ov.loc[list(E.SPLITS), "videos"] / ov.loc["ALL", "videos"]).round(3).to_dict())
print("Signer chồng lấn giữa split:", {k: len(v) for k, v in A.signer_overlap(m).items()}, "→ mong đợi toàn 0")

## A2. Phân bố theo gloss (class) — mất cân bằng, long-tail, số signer / class

In [ ]:
gt = E.gloss_table(m); bal = E.balance_stats(gt)
display(bal); save_tbl(bal, "A2_gloss_balance"); save_tbl(gt, "A2_gloss_table")
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
gt.total.plot.hist(bins=40, ax=ax[0]); ax[0].set_title("Số video / gloss (tất cả split)")
gt[list(E.SPLITS)].plot.box(ax=ax[1]); ax[1].set_title("Video / gloss theo split")
gt.signers_train.plot.hist(bins=30, ax=ax[2]); ax[2].set_title("Số signer khác nhau / gloss (train)")
save_fig(fig, "A2_gloss_distribution"); plt.show()
print("Gloss nhiều mẫu nhất:"); display(gt.head(5))
print("Gloss ít mẫu nhất:"); display(gt.tail(5))

## A3. Phân bố theo signer — mức tập trung, đa dạng, độ phủ gloss

In [ ]:
st = E.signer_table(m); save_tbl(st, "A3_signer_table"); display(st.head(8)); display(st.tail(4))
top5 = st.videos.head(5).sum() / len(m)
print(f"Top-5 signer đóng góp {top5:.1%} video | signer/split:", st.groupby("split").size().to_dict())
print("Số video / signer: min={}, median={:.0f}, max={}".format(st.videos.min(), st.videos.median(), st.videos.max()))
fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
s2 = st.sort_values("videos", ascending=False)
ax[0].bar(range(len(s2)), s2.videos, color=[COL[s] for s in s2.split]); ax[0].set_title("Video / signer (màu = split)")
ax[0].legend(handles=[Patch(color=c, label=k) for k, c in COL.items()]); ax[0].set_xlabel("signer (sắp xếp giảm dần)")
for k, c in COL.items():
    d = st[st.split == k]; ax[1].scatter(d.glosses, d.videos, color=c, label=k, s=22)
ax[1].set_xlabel("số gloss signer đã ký"); ax[1].set_ylabel("số video"); ax[1].set_title("Độ phủ gloss của từng signer"); ax[1].legend()
st.gloss_coverage.plot.hist(bins=20, ax=ax[2]); ax[2].set_title("Tỉ lệ gloss mỗi signer đã ký")
save_fig(fig, "A3_signers"); plt.show()

## A4. Số lần một signer lặp lại cùng một gloss

In [ ]:
pr = E.pair_repeats(m); display(pr.to_frame()); save_tbl(pr.to_frame(), "A4_pair_repeats")
fig, ax = plt.subplots(figsize=(5, 3)); pr.plot.bar(ax=ax); ax.set_xlabel("số video / (signer, gloss)"); ax.set_title("Lặp lại của cùng signer & gloss")
save_fig(fig, "A4_pair_repeats"); plt.show()
print(f"Cặp (signer, gloss) phổ biến nhất có {pr.idxmax()} video ({pr.max() / pr.sum():.1%} số cặp); tối đa {pr.index.max()} video/cặp.")

## A5. Tên gloss: biến thể (số đuôi), nhiều từ, mã ASL-LEX

In [ ]:
digit = gt.index.str.contains(r"\d+$").mean(); multi = gt.index.str.contains(r"[_\s]").mean()
print(f"Gloss có số đuôi (biến thể, vd BREAKFAST1): {digit:.1%} | nhiều từ/gạch dưới: {multi:.1%} | độ dài tên trung vị: {gt.index.to_series().str.len().median():.0f} ký tự")
grp = gt.groupby("gloss_base").size().sort_values(ascending=False)
print("Nhóm gloss cùng gốc (dễ nhầm lẫn):", int((grp > 1).sum()), "nhóm; lớn nhất:"); display(grp.head(10).to_frame("n_variants"))
print("Mã ASL-LEX:", E.lex_code_check(m))
save_tbl(grp.to_frame("n_variants"), "A5_gloss_variant_groups")

## A6. Cần bao nhiêu class? — số mẫu/class còn lại khi tăng K

In [ ]:
cc = E.coverage_curve(gt); display(cc); save_tbl(cc, "A6_coverage_curve", index=False)
eg = E.eligibility_grid(gt); display(eg); save_tbl(eg, "A6_eligibility", index=False)
fig, ax = plt.subplots(figsize=(7, 3.8))
for c in ["min_train", "median_train", "min_val", "min_test"]: ax.plot(cc.K, cc[c], marker="o", label=c)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("K class"); ax.set_ylabel("video / class"); ax.legend()
ax.set_title("Kịch bản tốt nhất: số mẫu/class khi chọn K class"); save_fig(fig, "A6_coverage_curve"); plt.show()

## A7. Dung lượng file (từ chỉ mục ZIP — không cần tải video)

In [ ]:
mb = m.file_size / 1e6
print(f"Tổng {m.file_size.sum()/1e9:.1f} GB | MB/video: p5={mb.quantile(.05):.2f}, p50={mb.median():.2f}, p95={mb.quantile(.95):.2f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for k, c in COL.items(): m.loc[m.split == k, "file_size"].div(1e6).plot.hist(bins=50, ax=ax[0], alpha=.5, color=c, label=k)
ax[0].set_xlabel("MB / video"); ax[0].legend(); ax[0].set_title("Kích thước file theo split")
st.mb_mean.plot.hist(bins=20, ax=ax[1]); ax[1].set_title("MB trung bình / signer (khác nhau → khác camera/độ phân giải?)")
save_fig(fig, "A7_file_size"); plt.show()

# Phần B — Mẫu video phân tầng theo signer
Chỉ tải ~1.5k video (chia đều mọi signer) để đo fps, độ phân giải, độ dài và MediaPipe. Video mẫu nằm ở thư mục tạm, chỉ giữ lại bảng thống kê.

In [ ]:
SS = None
if CFG["run_part_b"]:
    S = E.stratified_sample(m, CFG["sample_videos"], CFG["seed"])
    gb = S.file_size.sum() / 1e9
    print(f"Mẫu: {len(S)} video từ {S.signer_id.nunique()} signer, ≈{gb:.2f} GB (giới hạn {CFG['max_sample_gb']} GB)")
    assert gb <= CFG["max_sample_gb"], "Mẫu vượt max_sample_gb — giảm sample_videos"
    tmp = Path(tempfile.gettempdir()); sample_dir = tmp / "asl_sample_videos"
    cache_csv = OUT / "B_sample_stats.csv"
    if cache_csv.exists() and CFG["run_probe"] and pd.read_csv(cache_csv).det_rate.notna().sum() == 0:
        cache_csv.unlink()   # cache cũ không có kết quả MediaPipe → tính lại
    done = set(pd.read_csv(cache_csv).video_id) if cache_csv.exists() else set()
    need = S[~S.video_id.isin(done)]
    if len(need):
        log = A.download_videos(A.DEFAULT_CFG["zip_src"], need[["zip_name", "file_size"]], sample_dir, CFG["workers"],
                                log_path=OUT / "B_sample_download_log.csv")
        print("tải:", {k: int(v) for k, v in log.status.value_counts().items()})
        assert (log.status != "failed").all(), "Có video lỗi — chạy lại cell này"
    probe = A.make_probe(tmp / "asl_models") if CFG["run_probe"] else None
    SS = E.sample_stats(S, sample_dir, probe, CFG["probe_frames"], cache_csv)
    SS.to_csv(OUT / "B_sample_full.csv", index=False)
    print("Video lỗi:", int((~SS.ok.astype(bool)).sum()), "/", len(SS))
else:
    print("run_part_b=False → bỏ qua phần B")

In [ ]:
if SS is not None:
    ok = SS[SS.ok.astype(bool)]
    display(ok[["fps", "width", "height", "duration", "n_frames"]].describe(percentiles=[.05, .5, .95]).T.round(2))
    res = ok.groupby(["width", "height"]).size().sort_values(ascending=False).rename("videos").head(8); display(res.to_frame())
    ux = ok.assign(cfgv=ok.width.astype(int).astype(str) + "x" + ok.height.astype(int).astype(str) + "@" + ok.fps.round().astype(int).astype(str)).groupby("signer_id").cfgv.nunique()
    print(f"Số cấu hình (w,h,fps) khác nhau mỗi signer: median={ux.median():.0f}, max={ux.max()} → {int((ux > 1).sum())}/{len(ux)} signer dùng >1 cấu hình")
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
    for k, c in COL.items(): ok.loc[ok.split == k, "duration"].plot.hist(bins=30, ax=ax[0], alpha=.5, color=c, label=k)
    ax[0].set_title("Độ dài video (s) theo split"); ax[0].legend()
    ok.fps.round().value_counts().sort_index().plot.bar(ax=ax[1]); ax[1].set_title("FPS")
    ax[2].scatter(ok.file_size / 1e6, ok.duration, s=6, alpha=.4); ax[2].set_xlabel("MB"); ax[2].set_ylabel("độ dài (s)"); ax[2].set_title("File size vs độ dài")
    save_fig(fig, "B1_video_properties"); plt.show()
    d = ok.duration
    print(f"Độ dài: p50={d.median():.2f}s p90={d.quantile(.9):.2f}s p95={d.quantile(.95):.2f}s → gợi ý khung thời gian T ở notebook 02 (sau khi trim đoạn tay nghỉ sẽ ngắn hơn)")

In [ ]:
if SS is not None:
    ok = SS[SS.ok.astype(bool)]
    # Ước lượng độ dài cho TOÀN dataset từ file size (hồi quy tuyến tính trên mẫu)
    x, y = ok.file_size / 1e6, ok.duration
    a, b = np.polyfit(x, y, 1); r2 = 1 - ((y - (a * x + b)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    m["est_duration"] = np.clip(a * (m.file_size / 1e6) + b, 0, None)
    print(f"Hồi quy độ dài ≈ {a:.2f}·MB + {b:.2f}  (R²={r2:.2f}). R² thấp → bitrate/độ phân giải khác nhau, đừng tin con số ước lượng.")
    if r2 >= 0.5:
        display(m.groupby("split").est_duration.describe(percentiles=[.5, .9, .95]).round(2))
    if ok.det_rate.notna().any():
        sg = ok.groupby("signer_id").agg(split=("split", "first"), n=("video_id", "size"), det_rate=("det_rate", "mean"),
                                         two_hand_rate=("two_hand_rate", "mean"), motion=("motion_wrist", "median")).sort_values("det_rate")
        save_tbl(sg, "B2_probe_by_signer"); display(sg.head(8))
        fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
        ax[0].bar(range(len(sg)), sg.det_rate, color=[COL[s] for s in sg.split]); ax[0].axhline(.85, ls="--", c="r")
        ax[0].set_title("Tỉ lệ frame MediaPipe thấy tay, theo signer"); ax[0].set_xlabel("signer (tăng dần)")
        ok.det_rate.plot.hist(bins=20, ax=ax[1]); ax[1].set_title("Tỉ lệ thấy tay mỗi video")
        save_fig(fig, "B2_probe"); plt.show()
        print(f"Det rate toàn mẫu: {ok.det_rate.mean():.1%} | signer dưới 0.85: {int((sg.det_rate < .85).sum())}/{len(sg)}")
    else:
        print("Không có kết quả probe MediaPipe (xem cảnh báo ở cell trước).")

# Phần C — Báo cáo tự động + gói tải về

In [ ]:
def _s(df): return "```\n" + df.to_string() + "\n```"
L = ["# ASL Citizen — Full EDA report (tự động)", "",
     f"Nguồn bảng: {SRC}. Tổng {len(m):,} video, {m.gloss.nunique():,} gloss, {m.signer_id.nunique()} signer, {m.file_size.sum()/1e9:.1f} GB.", "",
     "## Tổng quan theo split", _s(ov), "",
     "## Cân bằng class (video/gloss)", _s(bal), "",
     f"- Median train/gloss = {gt.train.median():.0f}; val/gloss = {gt.val.median():.0f}; test/gloss = {gt.test.median():.0f}.",
     f"- Gloss có ≤10 video train: {(gt.train <= 10).mean():.1%}. Signer/gloss (train) trung vị = {gt.signers_train.median():.0f}.", "",
     "## Signer", f"- Top-5 signer = {top5:.1%} video. Video/signer: min {st.videos.min()}, median {st.videos.median():.0f}, max {st.videos.max()}.",
     f"- Signer/split: {st.groupby('split').size().to_dict()}.", "",
     "## Số mẫu/class khi chọn K class (kịch bản tốt nhất)", _s(cc), ""]
if SS is not None:
    okc = SS[SS.ok.astype(bool)]
    L += ["## Mẫu video phân tầng", f"- {len(okc)}/{len(SS)} video đọc được; fps median {okc.fps.median():.0f}; "
          f"độ phân giải phổ biến: {okc.groupby(['width','height']).size().idxmax()}; độ dài p50/p90 = {okc.duration.median():.2f}/{okc.duration.quantile(.9):.2f}s."]
    if okc.det_rate.notna().any():
        L += [f"- MediaPipe thấy tay: {okc.det_rate.mean():.1%} frame (trung bình)."]
L += ["", "## Câu hỏi để chọn pipeline (điền sau khi cả nhóm đọc)",
      "1. Dùng bao nhiêu class cho mốc chính? (xem đường cong K)", "2. Chọn mô hình/val thế nào khi val chỉ có vài signer? (CV theo signer?)",
      "3. Signer nào/điều kiện quay nào làm MediaPipe kém? Có cần lọc không?", "4. Chuẩn hoá & độ dài chuỗi T chọn theo phân bố độ dài ở trên."]
(OUT / "full_eda_report.md").write_text("\n".join(L), encoding="utf-8")
zp = shutil.make_archive(str(P.metrics / "full_eda_bundle"), "zip", OUT)
print("Đã ghi:", OUT / "full_eda_report.md"); print("Gói tải về:", zp, f"({Path(zp).stat().st_size/1e6:.1f} MB)")

## Lấy kết quả về máy
- **Kaggle:** panel *Output* → `asl_outputs/full_eda_bundle.zip` (hoặc `asl_outputs/full_eda/`) → Download. Muốn lưu lâu dài: *Save Version*.
- **Local:** `outputs/metrics/full_eda/` (bảng, `plots/`, `full_eda_report.md`) và `full_eda_bundle.zip`.
- Bảng dùng chung cho cả nhóm: `meta_full.csv` (mọi video + signer + split + kích thước), `A2_gloss_table.csv`, `A3_signer_table.csv`.